# 02 - Data inspection

Phase 3 check: look at what the loaders actually produce before training on it.

1. What is in the data (images, captions, questions) and how it is distributed.
2. How one training example is rendered, and which tokens carry loss.
3. What a padded batch looks like (right-padded for training, left-padded for evaluation).
4. End to end: a real batch through the VLM, with images on the fly vs the cached ViT features.

In [ ]:
import os, sys, json, time, collections
ROOT = os.path.abspath("..")
sys.path.insert(0, ROOT); os.chdir(ROOT)

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from PIL import Image
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from src.data.caption_dataset import CaptionDataset
from src.data.vqa_dataset import VQADataset
from src.data.collator import make_collator, batch_to_device
from src.data.feature_cache import FeatureStore
from src.data.manifest_dataset import read_jsonl
from src.data.templates import IGNORE_INDEX, IMAGE_PAD, get_image_token_id
from src.models.vit.load_pretrained import clip_transform

BLUE, ORANGE, GRID = "#2f6db5", "#d98324", "#e3e5e8"
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#8a8f98", "axes.labelcolor": "#4b5058", "xtick.color": "#4b5058", "ytick.color": "#4b5058",
})
M, N = "data/manifests/", 64
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
tf = clip_transform(224)
data = {n: read_jsonl(f"{M}{n}.jsonl") for n in ("caption_train", "caption_eval", "vqa_train", "vqa_eval")}
print(json.dumps({k: len(v) for k, v in data.items()}))
print("meta:", json.load(open(f"{M}meta.json"))["counts"])

## 1. The data

In [ ]:
rng = np.random.default_rng(3)
picks = rng.choice(len(data["caption_train"]), 4, replace=False)
fig, ax = plt.subplots(1, 4, figsize=(14, 4.4))
for a, i in zip(ax, picks):
    r = data["caption_train"][i]
    a.imshow(Image.open("data/" + r["image"])); a.axis("off")
    a.set_title("\n".join(__import__("textwrap").wrap(r["captions"][0], 34)), fontsize=8)
plt.tight_layout(); plt.show()
print("the 5 captions of the first image:"); [print(" -", c) for c in data["caption_train"][picks[0]]["captions"]]

In [ ]:
import textwrap
picks = rng.choice(len(data["vqa_train"]), 4, replace=False)
fig, ax = plt.subplots(1, 4, figsize=(14, 4.4))
for a, i in zip(ax, picks):
    r = data["vqa_train"][i]
    a.imshow(Image.open("data/" + r["image"])); a.axis("off")
    a.set_title("\n".join(textwrap.wrap(f"Q: {r['question']}", 34)) + f"\nA: {r['answer']}  [{r['answer_type']}]", fontsize=8)
plt.tight_layout(); plt.show()

### Distributions

In [ ]:
def share(rows, key):
    c = collections.Counter(r[key] for r in rows); n = sum(c.values())
    return {k: v / n for k, v in c.items()}

types = ["yes/no", "number", "other"]
tr, ev = share(data["vqa_train"], "answer_type"), share(data["vqa_eval"], "answer_type")
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))

x = np.arange(len(types)); w = 0.38
ax[0].bar(x - w/2, [tr[t] for t in types], w, color=BLUE, label=f"train ({len(data['vqa_train']):,})")
ax[0].bar(x + w/2, [ev[t] for t in types], w, color=ORANGE, label=f"eval ({len(data['vqa_eval']):,})")
ax[0].set_xticks(x); ax[0].set_xticklabels(types); ax[0].set_ylabel("share of questions")
ax[0].set_title("VQA answer type", fontsize=10, loc="left"); ax[0].legend(frameon=False, fontsize=8)

top = collections.Counter(r["answer"] for r in data["vqa_train"]).most_common(10)
ax[1].barh([a for a, _ in top][::-1], [c / len(data["vqa_train"]) for _, c in top][::-1], color=BLUE)
ax[1].set_title("Top-10 answers (train)", fontsize=10, loc="left"); ax[1].set_xlabel("share of questions")

cap_len = [len(c.split()) for r in data["caption_train"] for c in r["captions"]]
ax[2].hist(cap_len, bins=range(3, 31), color=BLUE)
ax[2].set_title("Caption length (words, train)", fontsize=10, loc="left"); ax[2].set_xlabel("words")
for a in ax: a.grid(axis="y" if a is not ax[1] else "x", color=GRID, lw=0.6); a.set_axisbelow(True)
plt.tight_layout(); plt.show()

ans_words = [len(r["answer"].split()) for r in data["vqa_train"]]
print(f"answers: {np.mean([w == 1 for w in ans_words]):.0%} are one word; max {max(ans_words)} words")
print(f"caption length: mean {np.mean(cap_len):.1f} words, min {min(cap_len)}, max {max(cap_len)}")
print("top-10 answers cover", f"{sum(c for _, c in top) / len(data['vqa_train']):.0%}", "of train questions (language prior a model can exploit)")
print("top question types:", collections.Counter(r["question_type"] for r in data["vqa_train"]).most_common(6))

## 2. One rendered training example

Training examples are ChatML conversations. The `<image>` marker becomes `<|vision_start|>` + 64 x `<|image_pad|>` + `<|vision_end|>`, and the loss is computed only on the assistant reply (plus its closing `<|im_end|>`).

In [ ]:
def render(item):
    ids, labels = item["input_ids"], item["labels"]
    runs, start = [], 0
    for i in range(1, len(ids) + 1):
        if i == len(ids) or (labels[i] != IGNORE_INDEX) != (labels[start] != IGNORE_INDEX):
            runs.append((labels[start] != IGNORE_INDEX, tok.decode(ids[start:i]))); start = i
    for trained, text in runs:
        text = text.replace(IMAGE_PAD * N, f"{IMAGE_PAD} x{N}")
        print(("TRAINED  " if trained else "masked   ") + repr(text))

cap = CaptionDataset(M + "caption_train.jsonl", "data", tok, N, transform=tf, sample="first")
vqa = VQADataset(M + "vqa_train.jsonl", "data", tok, N, transform=tf)
print("--- caption example"); render(cap[0])
print("\n--- VQA example"); render(vqa[0])
print(f"\nsequence length {len(vqa[0]['input_ids'])} of which {len(vqa[0]['input_ids']) - vqa[0]['input_ids'].count(get_image_token_id(tok))} text, {N} image; trained-on tokens: {sum(l != IGNORE_INDEX for l in vqa[0]['labels'])}")

## 3. Batches

Left: a right-padded training batch (what the loss sees). Right: a left-padded evaluation batch (prompt only, so `generate()` can continue from the last position).

In [ ]:
vqa_eval = VQADataset(M + "vqa_eval.jsonl", "data", tok, N, transform=tf, mode="eval")
train_b = make_collator(tok, "train")([vqa[i] for i in range(10)])
eval_b = make_collator(tok, "eval")([vqa_eval[i] for i in range(10)])

def layout(b):
    z = np.zeros(b["input_ids"].shape, dtype=int)                       # 0 padding
    z[b["attention_mask"].bool().numpy()] = 1                           # 1 context, loss-free
    z[(b["labels"] != IGNORE_INDEX).numpy()] = 2                        # 2 trained on
    return z

cmap = ListedColormap([GRID, "#9bb7d4", ORANGE])
fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
for a, b, title in [(ax[0], train_b, "train batch: right-padded"), (ax[1], eval_b, "eval batch: left-padded")]:
    a.imshow(layout(b), cmap=cmap, aspect="auto", interpolation="nearest", vmin=0, vmax=2)
    a.set_title(title, fontsize=10, loc="left"); a.set_xlabel("position"); a.set_ylabel("sample")
ax[1].legend(handles=[Patch(color=GRID, label="padding"), Patch(color="#9bb7d4", label="context (no loss)"), Patch(color=ORANGE, label="trained on")],
             frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=3)
plt.tight_layout(); plt.show()
print("train batch:", {k: tuple(v.shape) for k, v in train_b.items() if torch.is_tensor(v)})
print("eval batch last column all real tokens:", bool(eval_b["attention_mask"][:, -1].all()))

## 4. End to end: a real batch through the VLM

Same 8 training examples, once with images loaded and encoded on the fly, once from the cached ViT features. With the same (untrained) projector the loss should agree up to the fp16 rounding of the cache.

In [ ]:
from src.models.language_model import LanguageModel
from src.models.vit.load_pretrained import load_clip_vit
from src.models.vlm import MiniVLM
from src.utils.config import get_device

dev = get_device("auto")
torch.manual_seed(0)
vit = load_clip_vit().to(dev)
lm = LanguageModel.from_pretrained(dtype="float32", device=dev)
vlm = MiniVLM.build(vit, lm, "linear", N).to(dev).set_stage("align")
print("device:", dev, "| trainable params:", {k: f"{v[0]/1e6:.2f}M" for k, v in vlm.parameter_summary().items()})

In [ ]:
store = FeatureStore("data/features/clip_b16_l-2_224")
pix_ds = VQADataset(M + "vqa_train.jsonl", "data", tok, N, transform=tf)
feat_ds = VQADataset(M + "vqa_train.jsonl", "data", tok, N, feature_store=store)
collate = make_collator(tok, "train")
idx = list(range(100, 108))

def run(ds, backward=False):
    b = batch_to_device(collate([ds[i] for i in idx]), dev); b.pop("meta")
    vlm.zero_grad(set_to_none=True)
    t = time.time()
    out = vlm(**b)
    if backward: out.loss.backward()
    if dev.type == "mps": torch.mps.synchronize()
    return out, time.time() - t, b

out_pix, t_pix, bp = run(pix_ds, backward=True)
grad_pix = torch.cat([p.grad.flatten() for p in vlm.projector.parameters()]).clone()
out_feat, t_feat, bf = run(feat_ds, backward=True)
grad_feat = torch.cat([p.grad.flatten() for p in vlm.projector.parameters()]).clone()

print("batch keys  pixel path :", sorted(k for k in bp if torch.is_tensor(bp[k])))
print("batch keys  cached path:", sorted(k for k in bf if torch.is_tensor(bf[k])))
print(f"loss  pixels {out_pix.loss.item():.5f} | cached {out_feat.loss.item():.5f} | diff {abs(out_pix.loss.item() - out_feat.loss.item()):.1e}")
print(f"max |logit diff| {(out_pix.logits - out_feat.logits).abs().max().item():.3f} (logit scale ~{out_pix.logits.abs().mean().item():.1f})")
print(f"projector grad cosine similarity {torch.nn.functional.cosine_similarity(grad_pix, grad_feat, dim=0).item():.6f}")
print(f"forward+backward incl. data prep: pixel path {t_pix:.2f}s | cached path {t_feat:.2f}s (includes image decode + ViT only for pixels)")
print("only projector has gradients:", all(p.grad is None for p in lm.parameters()) and all(p.grad is None for p in vit.parameters()))

In [ ]:
# evaluation batch (left-padded): generation runs end to end from cached features
eval_feat = VQADataset(M + "vqa_eval.jsonl", "data", tok, N, feature_store=store, mode="eval")
b = batch_to_device(make_collator(tok, "eval")([eval_feat[i] for i in range(6)]), dev); meta = b.pop("meta")
vlm.eval()
gen = vlm.generate(b["input_ids"], b["attention_mask"], vision_features=b["vision_features"], max_new_tokens=6)
print("generated shape:", tuple(gen.shape))
for m_, g in zip(meta[:4], gen[:4]):
    print(f"  Q: {m_['question'][:50]:50} | gold: {m_['answer']:5} | untrained model: {tok.decode(g, skip_special_tokens=True)!r}")

## Summary

- Images, captions and questions load as expected; the label mask covers exactly the assistant reply.
- Padding is correct on both sides, and the loss ignores padding.
- A real batch runs through the VLM from both the on-the-fly and the cached path with matching loss, logits and gradients; only the projector receives gradients.
- Untrained outputs are unrelated to the questions, which is the baseline stage 1 and stage 2 must improve on.